# P2 on the published held-out pairs

**P2 is the model selected from the training experiments.** P0 and P1 are reference baselines for interpreting the held-out results.

| Model | Opening features, as signed A-minus-B differences |
| --- | --- |
| P0 | Word count |
| P1 | Mean Zipf frequency among listed content-word occurrences |
| P2 | Word count and mean content-word Zipf frequency |

The evaluation refits each frozen specification on all archived training pairs. It uses the original standardization and L2 logistic regression pipeline with `C=1`. The fitted models then score the published held-out pairs. A/B orientation uses the existing seeded assignment rule. Pairwise accuracy predicts A at probability 0.5 or higher.

The exact archived training input is checked against the saved P2 fingerprint. Word count and content-word Zipf use their existing extraction rules. All models use the same complete held-out pairs. The script checks for overlapping pairs, opening IDs, and OP discussions across splits. See the [standalone evaluation script](evaluate_p2_heldout_with_p0_p1_baselines.py) and the [original training comparison](compare_p0_p1_p2_word_count_and_content_word_zipf.ipynb).

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

REPO = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "team-setup").is_dir() and (p / "pyproject.toml").is_file()
)
RUNS = REPO / "project/data/processed/linguistic-features/paired-heldout-evaluation"
RUN_ID = None  # Set an explicit timestamp to revisit a particular saved run.
runs = sorted(RUNS.glob("*/results.json"))
if RUN_ID is None and not runs:
    raise FileNotFoundError("Run evaluate_p2_heldout_with_p0_p1_baselines.py first.")
report_path = RUNS / RUN_ID / "results.json" if RUN_ID else runs[-1]
report = json.loads(report_path.read_text())
predictions = pd.read_parquet(report_path.parent / "heldout_predictions.parquet")
assert report["split"] == "heldout"
assert len(predictions) == report["pairs"] and predictions["pair_id"].is_unique
assert predictions["post_id"].nunique() == report["discussions"]

print(f"Training pairs used to fit each model: {report['training_pairs']:,}")
print(f"Held-out pairs analyzed: {report['pairs']:,} of {report['published_pairs']:,}")
print(f"Held-out discussions analyzed: {report['discussions']:,} of {report['published_discussions']:,}")
print(f"A-wins prevalence: {report['a_win_prevalence']:.1%}")
print(f"Majority-label accuracy: {report['majority_accuracy']:.1%}")
if report["exclusions"]:
    display(pd.Series(report["exclusions"], name="Excluded held-out pairs").to_frame())

Training pairs used to fit each model: 3,451
Held-out pairs analyzed: 807 of 807
Held-out discussions analyzed: 542 of 542
A-wins prevalence: 50.9%
Majority-label accuracy: 50.9%


## Held-out performance

Intervals are percentile intervals from resampling whole held-out OP discussions, with identical weights for all models. The predictions remain fixed, so the intervals reflect which discussions were evaluated. ROC-AUC measures ranking, average precision is relative to the A-wins prevalence, and lower log loss indicates better probability estimates. These results apply to choosing the winning opening within a published matched pair.

In [2]:
metric_labels = {
    "roc_auc": "ROC-AUC",
    "average_precision": "Average precision",
    "pairwise_accuracy": "Pairwise accuracy",
    "log_loss": "Log loss",
}
performance = pd.DataFrame({
    model: {metric_labels[key]: f"{score['value']:.4f} [{score['ci_95'][0]:.4f}, {score['ci_95'][1]:.4f}]"
            for key, score in metrics.items()}
    for model, metrics in report["models"].items()
})
display(performance)

increments = []
for baseline, metrics in report["p2_minus_baseline"].items():
    for key, score in metrics.items():
        low, high = score["ci_95"]
        increments.append({
            "Comparison": f"P2 minus {baseline}", "Metric": metric_labels[key],
            "Change": score["value"], "95% CI lower": low, "95% CI upper": high,
        })
display(pd.DataFrame(increments).set_index(["Comparison", "Metric"]).round(4))
print(f"Paired discussion-bootstrap draws: {report['bootstrap_draws']:,}")

,P0,P1,P2
ROC-AUC,"0.6343 [0.5941, 0.6729]","0.6207 [0.5785, 0.6620]","0.6821 [0.6449, 0.7173]"
Average precision,"0.6469 [0.5974, 0.6961]","0.6125 [0.5604, 0.6647]","0.6843 [0.6366, 0.7318]"
Pairwise accuracy,"0.5923 [0.5567, 0.6272]","0.5911 [0.5554, 0.6253]","0.6468 [0.6141, 0.6797]"
Log loss,"0.6630 [0.6440, 0.6847]","0.6707 [0.6544, 0.6860]","0.6416 [0.6196, 0.6668]"


Change  95% CI lower  95% CI upper
Comparison  Metric                                               
P2 minus P0 ROC-AUC            0.0478        0.0176        0.0751
            Average precision  0.0375        0.0101        0.0642
            Pairwise accuracy  0.0545        0.0190        0.0888
            Log loss          -0.0214       -0.0367       -0.0076
P2 minus P1 ROC-AUC            0.0613        0.0281        0.0962
            Average precision  0.0719        0.0289        0.1131
            Pairwise accuracy  0.0558        0.0250        0.0862
            Log loss          -0.0291       -0.0475       -0.0079

Paired discussion-bootstrap draws: 1,000


## Accuracy as a count

Each held-out pair contributes one pick. The model picks side A at a predicted probability of 0.5 or higher.

In [3]:
accuracy_rows = []
for model in report["models"]:
    chosen_a = predictions[f"{model}_probability_a_wins"].ge(0.5)
    correct = int(chosen_a.eq(predictions["a_wins"]).sum())
    accuracy_rows.append({
        "Model": model, "Correct picks": correct, "Held-out pairs": len(predictions),
        "Accuracy (%)": 100 * correct / len(predictions),
    })
accuracy = pd.DataFrame(accuracy_rows).set_index("Model").round(1)
display(accuracy)

,Correct picks,Held-out pairs,Accuracy (%)
Model,,,
P0,478,807,59.2
P1,477,807,59.1
P2,522,807,64.7
